# R10 · TwinCheck demo for the video (Rathi)

**Runtime → T4 GPU** (or better), switch on **Notebook access** for `GITHUB_TOKEN` and `HF_TOKEN` in the 🔑 panel, then **Run all**.

Steps: 1 setup (set `VERDICT` = `"rule"` or `"head"`) → 2 tests → 3 run the 6 examples and push them (ends with `PUSHED`) → 4 launch the app (public `gradio.live` link; the cell keeps running, stop it when the recording is done).

R10 polish: Arabic claims and notes right-to-left, a confidence bar, citations [1]–[3] in the note link to their sources, and the footer says which verdict method is running. Script and shot list: `notes/R10_demo_video_script.md`.


In [ ]:
# 1. Setup
from google.colab import userdata
import os, json
REPO = '/content/twincheck'
if not os.path.exists(REPO):
    !git clone https://github.com/himaallu/Group-Project.git {REPO}
%cd {REPO}
!git config user.name "Rathi"
!git config user.email "335395537+rathiabeth-sudo@users.noreply.github.com"
!git fetch -q origin && git checkout -q main && git pull -q origin main

GH = userdata.get('GITHUB_TOKEN').strip()
!git remote set-url --push origin https://{GH}@github.com/himaallu/Group-Project.git
!pip install -q -U gradio bitsandbytes rank_bm25 faiss-cpu
!python -m spacy download -q en_core_web_trf   # entities for the verdict head
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()
from huggingface_hub import HfApi
USER = HfApi(token=os.environ['HF_TOKEN']).whoami()['name']
os.environ['TWINCHECK_NLI'] = 'MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7'   # zero-shot until R4 v2 wins
!python -m src.retrieval.build_index download --repo {USER}/twincheck-index-v1

# Which verdict the demo shows: "rule" (strongest entail vs contradict snippet; 5 of 6
# examples right in R6) or "head" (Himasri's trained head, 2 of 6). Agree it in the group.
VERDICT = "rule"
os.environ['TWINCHECK_VERDICT'] = 'none' if VERDICT == 'rule' else 'models/verdict_twin_s42'
!python -m src.pipeline --status

## 2. Tests

In [ ]:
!python -m pytest tests/demo tests/explain tests/test_pipeline.py -q

## 3. Run the 6 ready examples and push what the model says

In [ ]:
OUT = f'results/R10/examples_{VERDICT}.json'
!python -m src.demo.app --run-examples {OUT}
# PUSHED is printed only if the examples ran, and the commit and push worked
!git checkout -q -B rathi/R10-video && git add {OUT} && git commit -q -m "[R10] Demo example results ({VERDICT} verdict)" && git push -q -f origin rathi/R10-video && echo PUSHED

## 4. Launch the demo (click the `https://….gradio.live` link) and record. Stop the cell to close it.

In [ ]:
!python -m src.demo.app --share